# Drone env — browser eval

Pick a checkpoint, click **Build & load**. The recipe `just build-web <path>` bakes those weights into a WASM bundle and the iframe below reloads against it.

In [ ]:
import subprocess, atexit, time, pathlib, ipywidgets as W
from IPython.display import IFrame, display, clear_output

def find_project_root(start: pathlib.Path = None) -> pathlib.Path:
    start = start or pathlib.Path.cwd()
    for p in [start, *start.parents]:
        if (p / 'justfile').exists():
            return p
    raise RuntimeError(f'No justfile found above {start}')

PROJECT = find_project_root()
BUILD   = PROJECT / 'build' / 'web'
PORT    = 8765

BUILD.mkdir(parents=True, exist_ok=True)
srv = subprocess.Popen(
    ['python3', '-m', 'http.server', '-d', str(BUILD), str(PORT)],
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
)
atexit.register(srv.terminate)
time.sleep(0.3)

def list_checkpoints():
    base = PROJECT / 'checkpoints/drone'
    return sorted(base.rglob('*.bin'), key=lambda p: p.stat().st_mtime, reverse=True)

def rebuild(model_path):
    return subprocess.run(
        ['just', 'build-web', model_path],
        cwd=PROJECT, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
    )

print(f'Project root: {PROJECT}')
print(f'Server up on :{PORT}')

In [ ]:
ckpts = list_checkpoints()
if not ckpts:
    print('No checkpoints in checkpoints/drone/. Train one first: just train cpu hover')
else:
    def label(p, is_latest):
        run = p.parent.name
        try:
            step = f'{int(p.stem):,} steps'
        except ValueError:
            step = p.stem
        tag = '  ← latest' if is_latest else ''
        return f'{run} · {step}{tag}'

    options = [(label(p, i == 0), str(p)) for i, p in enumerate(ckpts)]
    picker = W.Dropdown(options=options, value=str(ckpts[0]),
                        description='Model:', layout=W.Layout(width='80%'))
    button = W.Button(description='Build & load', button_style='primary')
    status = W.Output()
    frame  = W.Output()

    def on_click(_):
        with status:
            clear_output(); print(f'Building with {picker.value} ...')
        res = rebuild(picker.value)
        with status:
            clear_output()
            if res.returncode != 0:
                print('Build failed:\n' + res.stdout); return
            print(f'Loaded: {picker.value}')
        with frame:
            clear_output()
            display(IFrame(f'/proxy/{PORT}/game.html?v={int(time.time())}', width=960, height=640))

    button.on_click(on_click)
    display(W.VBox([W.HBox([picker, button]), status, frame]))
    on_click(None)